<a href="https://colab.research.google.com/github/23SCSE1012098/-Intelligent-Resume-Screening-and-Job-Recommendation-System/blob/main/%F0%9F%A4%96_Intelligent_Resume_Screening_and_Job_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers pypdf nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 388.1/388.1 kB 5.9 MB/s eta 0:00:00


In [2]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.corpus import stopwords
from nltk import word_tokenize, sent_tokenize

from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

STOP_WORDS = set(stopwords.words("english"))
print("Libraries ready")

Libraries ready


In [3]:
from google.colab import files
uploaded = files.upload()

Saving resume (15).pdf to resume (15).pdf


In [4]:
import pypdf
import io

# Get the PDF filename and content from the uploaded dictionary
pdf_filename = list(uploaded.keys())[0]
pdf_content = uploaded[pdf_filename]

# Open the PDF file from its byte content
pdf_file = io.BytesIO(pdf_content)
pdf_reader = pypdf.PdfReader(pdf_file)

resume_text = ""
for page_num in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[page_num]
    resume_text += page.extract_text() + "\n"

# Create a DataFrame with the extracted text
# Assuming for a single uploaded PDF, we can assign a default ID and Category
resume_df = pd.DataFrame([
    {
        "ID": 1, # Placeholder ID
        "Category": "Uploaded_Resume", # Placeholder Category
        "resume_text": resume_text
    }
]).dropna(subset=["resume_text"]).reset_index(drop=True)

print(resume_df.shape)
display(resume_df.head())

(1, 3)


,ID,Category,resume_text
0,1,Uploaded_Resume,RAM KUMAR\n♂phone-alt+91 7248720634✉ramkumar72...


In [8]:
from google.colab import files
print("Please upload your job descriptions CSV file (e.g., job_descriptions.csv).")
uploaded_jobs = files.upload()

Please upload your job descriptions CSV file (e.g., job_descriptions.csv).


Saving Galgotias University Mail - Notice For Virtual Campus Drive 2027 - Amantya Technologies (1).pdf to Galgotias University Mail - Notice For Virtual Campus Drive 2027 - Amantya Technologies (1).pdf


In [6]:
import io
import pypdf

job_filename = list(uploaded_jobs.keys())[0]
job_content = uploaded_jobs[job_filename]

# Open the PDF file from its byte content
pdf_file = io.BytesIO(job_content)
pdf_reader = pypdf.PdfReader(pdf_file)

job_text = ""
for page_num in range(len(pdf_reader.pages)):
    page = pdf_reader.pages[page_num]
    job_text += page.extract_text() + "\n"

# Create a DataFrame with the extracted text
# Assuming for a single uploaded PDF, we can assign a default title and ID
job_df = pd.DataFrame([
    {
        "job_id": 1, # Placeholder ID
        "job_title": "Uploaded_Job_Description", # Placeholder title
        "job_text": job_text
    }
]).dropna(subset=["job_text"]).reset_index(drop=True)

print(job_df.shape)
display(job_df.head())

(1, 3)


,job_id,job_title,job_text
0,1,Uploaded_Job_Description,"Plot No. 2, Sector-17A, Yamuna Expressway, Gre..."


## 2. Text Preprocessing

To ensure accurate matching, we need to clean and preprocess the raw text data from both the resume and the job descriptions. This involves several steps:

1.  **Tokenization**: Breaking down the text into individual words or tokens.
2.  **Lowercasing**: Converting all text to lowercase to ensure consistency.
3.  **Stop Word Removal**: Eliminating common words (e.g., 'the', 'is', 'and') that do not carry significant meaning.
4.  **Alphanumeric Filtering**: Removing non-alphanumeric characters to reduce noise.

This preprocessing helps in standardizing the text, making it suitable for further analysis and feature extraction.

In [7]:
def preprocess_text(text):
    # Tokenize the text
    tokens = word_tokenize(text)
    # Convert to lowercase, remove stopwords and non-alphanumeric tokens
    cleaned_tokens = [
        token.lower()
        for token in tokens
        if token.lower() not in STOP_WORDS and token.isalnum()
    ]
    return " ".join(cleaned_tokens)

# Apply preprocessing to resume text
resume_df["cleaned_resume"] = resume_df["resume_text"].apply(preprocess_text)

# Apply preprocessing to job description text
job_df["cleaned_job_description"] = job_df["job_text"].apply(preprocess_text)

print("Preprocessing complete!")
display(resume_df.head())
display(job_df.head())

Preprocessing complete!


,ID,Category,resume_text,cleaned_resume
0,1,Uploaded_Resume,RAM KUMAR\n♂phone-alt+91 7248720634✉ramkumar72...,ram kumar summary computer science engineering...


,job_id,job_title,job_text,cleaned_job_description
0,1,Uploaded_Job_Description,"Plot No. 2, Sector-17A, Yamuna Expressway, Gre...",plot 2 yamuna expressway greater noida gautam ...


## 3. Embedding Generation and Similarity Calculation

To effectively compare the resume and job description, we need to convert their cleaned text into a numerical format that captures their semantic meaning. This is achieved through **embedding generation** using a pre-trained `SentenceTransformer` model.

Once we have the embeddings, we can calculate the **cosine similarity** between the resume embedding and each job description embedding. Cosine similarity measures the cosine of the angle between two non-zero vectors in a multi-dimensional space. A higher cosine similarity score indicates a greater degree of similarity between the text contents.

This step will provide a quantitative measure of how well a resume matches a given job description.

In [9]:
# Load a pre-trained Sentence Transformer model
# You can choose different models based on your needs, e.g., 'all-MiniLM-L6-v2', 'paraphrase-MiniLM-L3-v2'
model_name = 'all-MiniLM-L6-v2'
model = SentenceTransformer(model_name)

print(f"Sentence Transformer model '{model_name}' loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sentence Transformer model 'all-MiniLM-L6-v2' loaded successfully.


In [10]:
# Generate embeddings for the cleaned resume text
resume_embeddings = model.encode(resume_df["cleaned_resume"].tolist())

# Generate embeddings for the cleaned job descriptions
job_description_embeddings = model.encode(job_df["cleaned_job_description"].tolist())

print(f"Generated embeddings for {len(resume_embeddings)} resumes and {len(job_description_embeddings)} job descriptions.")

Generated embeddings for 1 resumes and 1 job descriptions.


In [11]:
# Calculate cosine similarity between the resume and job description embeddings
# Since we have one resume and one job description, we'll calculate the similarity between them
similarity_score = cosine_similarity(resume_embeddings, job_description_embeddings)[0][0]

print(f"Cosine Similarity Score: {similarity_score:.4f}")

# You can also add the similarity score to your job_df if you have multiple job descriptions
# For a single job description, it's a direct score.
job_df['similarity_score'] = similarity_score

display(job_df[['job_title', 'similarity_score']].head())

Cosine Similarity Score: 0.5502


,job_title,similarity_score
0,Uploaded_Job_Description,0.550209


## 4. Conclusion and Interpretation

We have successfully extracted text from the resume and job description PDFs, preprocessed the text, generated embeddings, and calculated the cosine similarity. The `similarity_score` indicates how well the resume content aligns with the job description requirements, with higher scores suggesting a better match.

### Interpretation of the Similarity Score:

*   **Score close to 1**: Indicates a very strong match, implying the resume contains many relevant keywords and concepts present in the job description.
*   **Score around 0.5-0.7**: Suggests a moderate match. There's some overlap, but potential areas for improvement in the resume to better align with the job description.
*   **Score close to 0**: Indicates a very weak or no significant match.

In [13]:
print(f"The resume has a similarity score of {job_df['similarity_score'].iloc[0]:.4f} with the job description: '{job_df['job_title'].iloc[0]}'.")

if job_df['similarity_score'].iloc[0] > 0.7:
    print("This is a strong match. The resume is highly aligned with the job description.")
elif job_df['similarity_score'].iloc[0] > 0.5:
    print("This is a moderate match. The resume aligns reasonably well, but there might be room for improvement to enhance the match.")
else:
    print("This is a weak match. Consider tailoring the resume more closely to the job description.")

The resume has a similarity score of 0.5502 with the job description: 'Uploaded_Job_Description'.
This is a moderate match. The resume aligns reasonably well, but there might be room for improvement to enhance the match.
